#### Import

In [1]:
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from itertools import product
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from tqdm import tqdm
from functions_utils import *
from functions_data import *
from functions_optimize import *
from functions_eval import *

S = 30
LEVEL = "high"
SEED = 1

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

BASE_PATH = "/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result"

✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=30
✅ 시뮬레이션 초기화 완료: S=30, Randomness='high', Random Seed=1, M1=722.00, M2=1957.00


#### Preliminaries

In [2]:
print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, obj_ind = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)
print("-"*100)

print("[Holistic Aggregation Model optimization]")
x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, obj_hol = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)
print("-"*100)

print("[Extracting target_i from Holistic Aggregation Model]")
x_extracted, ep_extracted, em_extracted, yp_extracted, ym_extracted, d_extracted, dp_extracted, dm_extracted, i_map_extracted = extracted_target(I, T, S, x_hol, ep_hol, em_hol, yp_hol, ym_hol, d_hol, dp_hol, dm_hol)
print("-"*100)

# print("[Holistic Aggregation Model optimization without target_i]")
# x_without, ep_without, em_without, yp_without, ym_without, d_without, dp_without, dm_without, i_map_without = optimize_without_loop(R, K, K0, P_DA, P_RT, P_PN, I, T, S)
# print("-"*100)

[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  20%|██        | 1/5 [00:00<00:00,  5.06it/s]

Optimal solution found for target_i=0! Objective value: 222190.54193013997
Set parameter MIPGap to value 1e-07


...

Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00,  5.25it/s]

Optimal solution found for target_i=3! Objective value: 454586.07160139777
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:00<00:00,  5.35it/s]

Optimal solution found for target_i=4! Objective value: 170336.98758043227
----------------------------------------------------------------------------------------------------
[Holistic Aggregation Model optimization]
Set parameter MIPGap to value 0.001


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.001

Optimize a model with 61350 rows, 57870 columns and 201750 nonzeros
Model fingerprint: 0xdfd64afc
Variable types: 43470 continuous, 14400 integer (14400 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [2e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+03]
Presolve removed 30876 rows and 24084 columns
Presolve time: 0.21s
Presolved: 30474 rows, 33786 columns, 101867 nonzeros
Variable types: 26389 continuous, 7397 integer (7397 binary)
Found heuristic solution: objective 1231669.6871
Found heuristic solution: objective 1263249.8317
Deterministic concurrent LP optimizer: primal and dual simplex
Showing primal log only...

Concurrent spin time: 0.00s

Solved with dual simplex
Extra simplex iterations after un

#### Price Functions

In [39]:
def calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='mean_all'):
    """
    다양한 방식으로 대표값을 계산하는 함수
    
    Parameters:
    - P_PN: [t, s] 차원의 PN 가격 데이터
    - P_RT: [t, s] 차원의 RT 가격 데이터  
    - dp_hol: [target_i, t, s] 차원의 dp 홀딩 데이터
    - dm_hol: [target_i, t, s] 차원의 dm 홀딩 데이터
    - method: 대표값 계산 방식
        - 'mean_all': 모든 값을 포함하는 평균
        - 'mean_nonzero': 0이 아닌 값에 대한 평균
        - 'median': 중앙값
        - 'max': 최대값
    
    Returns:
    - PN_rep: [t, s] 차원의 PN 대표값
    - RT_rep: [t, s] 차원의 RT 대표값
    - dp_rep: [target_i, t, s] 차원의 dp 대표값
    - dm_rep: [target_i, t, s] 차원의 dm 대표값
    """
    
    def apply_method_to_array(arr, method, axis=-1):
        """배열에 지정된 방식으로 대표값 계산"""
        if method == 'mean_all':
            return np.mean(arr, axis=axis, keepdims=True)
        
        elif method == 'mean_nonzero':
            # 0이 아닌 값에 대한 평균
            # axis=-1에 대해서만 처리 (시나리오 차원)
            result = np.zeros(arr.shape[:-1] + (1,))
            
            # 각 슬라이스에 대해 0이 아닌 값의 평균 계산
            if arr.ndim == 2:  # [t, s] 형태
                for t in range(arr.shape[0]):
                    nonzero_vals = arr[t, arr[t, :] != 0]
                    if len(nonzero_vals) > 0:
                        result[t, 0] = np.mean(nonzero_vals)
                    else:
                        result[t, 0] = 0
            elif arr.ndim == 3:  # [target_i, t, s] 형태
                for i in range(arr.shape[0]):
                    for t in range(arr.shape[1]):
                        nonzero_vals = arr[i, t, arr[i, t, :] != 0]
                        if len(nonzero_vals) > 0:
                            result[i, t, 0] = np.mean(nonzero_vals)
                        else:
                            result[i, t, 0] = 0
            return result
        
        elif method == 'median':
            return np.median(arr, axis=axis, keepdims=True)
        
        elif method == 'max':
            return np.max(arr, axis=axis, keepdims=True)
        
        else:
            raise ValueError(f"Unsupported method: {method}")
    
    # 각 배열에 대해 대표값 계산
    PN_rep = apply_method_to_array(P_PN, method)
    RT_rep = apply_method_to_array(P_RT, method)
    dp_rep = apply_method_to_array(dp_hol, method)
    dm_rep = apply_method_to_array(dm_hol, method)
    
    # s 차원을 원래 크기로 확장 (broadcasting을 위해)
    s_dim = P_PN.shape[-1]
    PN_rep = np.repeat(PN_rep, s_dim, axis=-1)
    RT_rep = np.repeat(RT_rep, s_dim, axis=-1)
    dp_rep = np.repeat(dp_rep, s_dim, axis=-1)
    dm_rep = np.repeat(dm_rep, s_dim, axis=-1)
    
    return PN_rep, RT_rep, dp_rep, dm_rep

In [ ]:
# 모든 s에 대해 똑같이 가격&양 둘 다 대표값으로 대입

# PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='mean_all')
# PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='median')
# PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='max')

RP = np.full((I, T, S, 2), np.nan, dtype=object)
RM = np.full((I, T, S, 2), np.nan, dtype=object)
BP = np.zeros((I, T, S), dtype=int)
BM = np.zeros((I, T, S), dtype=int)
BIG_POS = 1e5; BIG_NEG = -1e5

for target_i, t, s in product(range(I), range(T), range(S)):
    RP[target_i, t, s] = [[0, PN_rep[t, s]], [dp_rep[target_i, t, s], BIG_NEG]]
    BP[target_i, t, s] = 2
    RM[target_i, t, s] = [[0, RT_rep[t, s]], [dm_rep[target_i, t, s], BIG_POS]]
    BM[target_i, t, s] = 2

In [78]:
# 가격 -> 대표값, 양에 대해서는 쌍으로 맞춰서 대입 (원래 holistic 기준 값이 있던 쪽에 대표값으로 대입, 한쪽은 0으로 제한, 둘 다 0이면 그냥 0)

# PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='mean_all')
# PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='mean_nonzero')
# PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='median')    
PN_rep, RT_rep, dp_rep, dm_rep = calculate_representative_values(P_PN, P_RT, dp_hol, dm_hol, method='max')

RP = np.full((I, T, S, 2), np.nan, dtype=object)
RM = np.full((I, T, S, 2), np.nan, dtype=object)
BP = np.zeros((I, T, S), dtype=int)
BM = np.zeros((I, T, S), dtype=int)
BIG_POS = 1e5; BIG_NEG = -1e5

for target_i, t, s in product(range(I), range(T), range(S)):
    if dp_hol[target_i, t, s] > 0:
        RP[target_i, t, s] = [[0, PN_rep[t, s]], [dp_rep[target_i, t, s], BIG_NEG]]
        BP[target_i, t, s] = 2

        RM[target_i, t, s] = [[0, BIG_POS]]
        BM[target_i, t, s] = 1

    elif dm_hol[target_i, t, s] > 0:
        RP[target_i, t, s] = [[0, BIG_NEG]]
        BP[target_i, t, s] = 1

        RM[target_i, t, s] = [[0, RT_rep[t, s]], [dm_rep[target_i, t, s], BIG_POS]]
        BM[target_i, t, s] = 2

    else:
        RP[target_i, t, s] = [[0, BIG_NEG]]
        BP[target_i, t, s] = 1

        RM[target_i, t, s] = [[0, BIG_POS]]
        BM[target_i, t, s] = 1

#### Stepwise Model

In [79]:
x_part, yp_part, ym_part, z_part, zc_part, zd_part, dp_part, dm_part, up_part, um_part, wp_part, wm_part, obj_part, RP_CLEARED, RM_CLEARED = stepwise_optimize_forall(I, T, S, R, K, K0, P_DA, P_RT, P_PN, RP, RM, BP, BM, M1)

Optimizing Stepwise for each target_i: 100%|██████████| 5/5 [00:01<00:00,  3.90it/s]


#### Results

In [80]:
target_i = 1
scen = 10

header = (
    f"{'s':>2} {'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
# print(header)

for s, t in product(range(scen,scen+1), range(T)):
    print(header)
    # individual
    # print(
    #     f"{s:>2} {t:>2} | "
    #     f"{R[target_i, t, s]:>8.2f} {x_ind[target_i][t]:>8.2f} {yp_ind[target_i][t, s]:>8.2f} {ym_ind[target_i][t, s]:>8.2f} "
    #     f"{0:>8.2f} {0:>8.2f} {zc_ind[target_i][t, s]:>8.2f} {zd_ind[target_i][t, s]:>8.2f} {z_ind[target_i][t, s]:>8.2f}"
    # )
    # stepwise
    print(
        f"{s:>2} {t:>2} | "
        f"{R[target_i, t, s]:>8.2f} {x_part[target_i][t]:>8.2f} {yp_part[target_i][t, s]:>8.2f} {ym_part[target_i][t, s]:>8.2f} "
        f"{dp_part[target_i][t, s]:>8.2f} {dm_part[target_i][t, s]:>8.2f} {zc_part[target_i][t, s]:>8.2f} {zd_part[target_i][t, s]:>8.2f} {z_part[target_i][t, s]:>8.2f}"
    )
    # holistic
    print(
        f"{s:>2} {t:>2} | "
        f"{R[target_i, t, s]:>8.2f} {x_hol[target_i, t]:>8.2f} {ep_hol[target_i, t, s]:>8.2f} {em_hol[target_i, t, s]:>8.2f} "
        f"{dp_hol[target_i, t, s]:>8.2f} {dm_hol[target_i, t, s]:>8.2f} {zc_hol[target_i, t, s]:>8.2f} {zd_hol[target_i, t, s]:>8.2f} {z_hol[target_i, t, s]:>8.2f}"
    )
    print()

 s  t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
10  0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
10  0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00

 s  t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
10  1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
10  1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00

 s  t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
10  2 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
10  2 |     0

In [81]:
compare_step_vs_hol(x_hol, ep_hol, em_hol, dp_hol, dm_hol, x_part, yp_part, ym_part, dp_part, dm_part, T, S, I)

STEPWISE vs HOLISTIC 모델 비교 (시나리오 평균)

타겟 참여자 0번:
--------------------------------------------------------------------------------------------------------------
 t |    x_hol   x_step |   yp_hol  yp_step |   ym_hol  ym_step |   dp_hol  dp_step |   dm_hol  dm_step
--------------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00
 1 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00
 2 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00
 3 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00
 4 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00
 5 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00 |     0.00     0.00
 6 |    

In [82]:
compare_holall_vs_holwithstep(x_hol, ep_hol, em_hol, dp_hol, dm_hol, x_part, yp_part, ym_part, dp_part, dm_part, T, S, I)

전체 holistic 최적화 vs 본인만 stepwise 최적화 + 나머지는 (본인포함) holistic
 i |  t |  x_hol_sum   x_part_sum |  ep_hol_mean  yp_part_sum |  em_hol_mean  ym_part_sum |  dp_hol_mean  dp_part_sum |  dm_hol_mean  dm_part_sum
-------------------------------------------------------------------------------------------------------------------------------------------------
 0 |  0 |       0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00
 0 |  1 |       0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00
 0 |  2 |       0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00
 0 |  3 |       0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00 |         0.00         0.00
 0 |  4 |       0.00         0.00 |         0.00         0.00 |  

In [83]:
# compare_holall_vs_stepsum(a_hol, bp_hol, bm_hol, dp_hol, dm_hol, x_part, yp_part, ym_part, dp_part, dm_part, T, S, I)